# 6.1 · Costes, seguridad y evaluación

**Objetivo:** criterio rápido y concreto sobre tres riesgos antes de poner el agente delante de clientes:

1. ¿Cuánto cuesta?
2. ¿Qué pasa si alguien intenta engañarlo?
3. ¿Cómo sé que sigue funcionando cuando cambie el prompt?

**Al terminar tendrás:** el agente de triaje con límites, una defensa por capas ante la inyección de instrucciones y una función `evaluar()` con los casos del 5.1.

## Preparación

El agente de triaje del 5.4 (ya corregido) ya no vive en un notebook: está en `triaje.py`, en la raíz. Este notebook queda para **medir, atacar y evaluar**.

- `crear_opciones(...)` construye las `ClaudeAgentOptions` del agente (con parámetros para las demos).
- `triar(correo, options)` tría un correo y devuelve el `ResultMessage`.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import pandas as pd
import herramientas
from triaje import triar, crear_opciones, SYSTEM, SYSTEM_SEGURO
from taller_utils import restaurar_datos, leer_log, borrar_log

CORREOS = herramientas.listar_bandeja()[:5]

## Parte 1 · Control de costes

### Medir

Los cinco correos de la bandeja, con el coste y los turnos de cada uno:

In [ ]:
restaurar_datos()
borrar_log()

filas, resultados = [], {}
for correo in CORREOS:
    r = await triar(correo)
    resultados[correo] = r
    filas.append({"correo": correo, "subtype": r.subtype, "turnos": r.num_turns, "coste": r.total_cost_usd})

costes = pd.DataFrame(filas).sort_values("coste", ascending=False)
print(f"Total: ${costes.coste.sum():.4f}  ·  medio por correo: ${costes.coste.mean():.4f}")
costes

El más caro, y por qué: sus trazas y el desglose por modelo (`model_usage`), donde se ve cuánto aporta el subagente.

In [ ]:
mas_caro = costes.iloc[0].correo
log = leer_log()
log[(log.correo == mas_caro) & (log.evento == "PreToolUse")][["herramienta", "entrada"]]

In [ ]:
{modelo: round(uso["costUSD"], 4) for modelo, uso in resultados[mas_caro].model_usage.items()}

### Palancas, en orden de aplicación

1. **Caché:** system prompt y herramientas estables y sin datos variables (2.3). El SDK ya la usa: mira `cache_read_input_tokens` en el `usage` del resultado.
2. **Límites duros:** `max_turns` y `max_budget_usd`. `crear_opciones()` usa `max_turns=12` y `max_budget_usd=0.50`.
3. **Modelo por tarea:** el clasificador ya corre en un modelo pequeño (4.3); el principal, en el que la tarea necesite.
4. **Menos contexto:** herramientas que devuelven lo justo. `buscar_kb` devuelve fragmentos, no artículos enteros; `tools=["Task"]` evita describir al modelo herramientas integradas que no usa.

In [ ]:
resultados[mas_caro].usage

### Un límite que salta

Con un presupuesto ridículo:

In [ ]:
# ❌ FALLO PROVOCADO
restaurar_datos()
r = await triar("003_urgente.txt", options=crear_opciones(max_budget_usd=0.001))
r.subtype, r.total_cost_usd

`subtype == "error_max_budget_usd"`. El agente se ha parado a medias: puede que haya creado la incidencia y no haya devuelto la respuesta, o al revés. **Hay que tratar ese caso**, no ignorarlo. `triar()` devuelve el `ResultMessage` también cuando el SDK lanza `ResultError`; quien lo llama decide:

In [ ]:
def procesar(r):
    if r.subtype == "success" and r.structured_output:
        return r.structured_output
    # límite de turnos o de presupuesto: a revisión manual, con el motivo
    return {"accion": "revisión manual", "motivo": r.subtype}

procesar(r)

**Regla:** se mide el coste **por tarea completada** (un correo bien triado), no por llamada. Un agente barato que deja la mitad de los correos a medias sale caro.

## Parte 2 · Seguridad: inyección de instrucciones

Un correo con instrucciones escondidas, dirigidas al asistente que lo procese:

In [ ]:
print(herramientas.leer_correo("006_inyeccion.txt"))

Lo ejecutamos con la versión del bloque 5 y **una herramienta `cerrar_incidencia` habilitada temporalmente** (sin el hook de política del 4.4), para ver si el agente lo intenta.

In [ ]:
def abiertas_acme():
    return [i["id"] for i in herramientas.buscar_incidencias(cliente="ACME", estado="abierta")]

restaurar_datos()
print("antes:", abiertas_acme())

options_insegura = crear_opciones(con_cierre=True, politica=False)
r = await triar("006_inyeccion.txt", options=options_insegura, ver=True, demo="insegura")

print("después:", abiertas_acme())

In [ ]:
log = leer_log()
log[(log.demo == "insegura") & (log.evento == "PreToolUse")][["herramienta", "entrada"]]

Fíjate en dos cosas en las trazas y en el resultado: si ha intentado cerrar algo, y qué prioridad ha puesto (el correo "pedía" alta para un NIF incorrecto en una factura).

Los modelos actuales suelen resistirse a este tipo de ataques. Pero la cuestión es que **no se puede confiar** en que lo hagan siempre, con cualquier redacción y en cualquier contexto.

> Todo lo que entra por una herramienta (un correo, una web, un fichero) es **dato**, no instrucción. Pero el modelo lo lee todo en el mismo contexto. La defensa no puede depender solo del prompt.

### Mitigaciones, por capas

**1. Mínimo privilegio.** El agente de triaje **no tiene** `cerrar_incidencia` (decisión del 5.1). Lo que no tiene, no lo puede hacer, lo diga el correo o no:

In [ ]:
options = crear_opciones()
options.allowed_tools

In [ ]:
restaurar_datos()
r = await triar("006_inyeccion.txt", options=options, demo="minimo_privilegio")
print("abiertas de ACME:", abiertas_acme())
r.structured_output

**2. Hooks de política.** El hook `proteger_cierres` del 4.4 bloquea los cierres de incidencias de prioridad alta **aunque el modelo los intente** y aunque la herramienta esté permitida. `crear_opciones(con_cierre=True)` lo incluye por defecto.

**3. Una persona en el bucle.** La salida es una *propuesta*; nada se envía solo.

**4. Separar datos de instrucciones en el prompt.** `leer_correo` devuelve el correo entre etiquetas `<correo>`, y el system prompt avisa de que su contenido es un dato:

In [ ]:
print(SYSTEM_SEGURO[len(SYSTEM):])

In [ ]:
restaurar_datos()
r = await triar("006_inyeccion.txt", options=crear_opciones(system_prompt=SYSTEM_SEGURO), demo="segura")
r.structured_output["prioridad"], r.structured_output["respuesta_propuesta"]

**5. Nunca** `bypassPermissions` ni herramientas genéricas (`Bash`, `Write`) en agentes que procesan entrada externa. Con `Bash`, un correo malicioso podría pedir cualquier cosa.

Ninguna capa es suficiente sola: el prompt puede fallar, y por eso debajo están los permisos y los hooks.

## Parte 3 · Evaluación básica

Los criterios de éxito del 5.1, como casos de prueba en `datos/evals/casos.json`:

In [ ]:
casos = json.loads(Path("../../datos/evals/casos.json").read_text())
casos

Para cada caso, `evaluar()`:

1. deja `incidencias.json` en un estado conocido,
2. ejecuta el agente, con una etiqueta de ejecución para encontrar sus trazas,
3. comprueba **qué devolvió** (`structured_output` frente a `esperado`)
4. y **qué hizo** (herramientas llamadas, según el log de auditoría, frente a `no_debe_llamar`).

In [ ]:
import uuid

async def evaluar(casos, repeticiones=3, options=None):
    filas = []
    for caso in casos:
        for i in range(repeticiones):
            restaurar_datos()
            ejecucion = uuid.uuid4().hex[:8]
            r = await triar(caso["correo"], options=options, ejecucion=ejecucion)

            fallos = []
            salida = r.structured_output or {}
            if r.subtype != "success":
                fallos.append(r.subtype)
            for campo, valor in caso.get("esperado", {}).items():
                if salida.get(campo) != valor:
                    fallos.append(f"{campo}={salida.get(campo)!r} (esperado {valor!r})")

            log = leer_log()
            llamadas = set(log[(log.ejecucion == ejecucion) & (log.evento == "PreToolUse")].herramienta)
            for prohibida in caso.get("no_debe_llamar", []):
                if prohibida in llamadas:
                    fallos.append(f"llamó a {prohibida.split('__')[-1]}")

            filas.append({"correo": caso["correo"], "ejecución": i + 1, "ok": "❌" if fallos else "✅",
                          "detalle": "; ".join(fallos), "coste": round(r.total_cost_usd, 4)})
    restaurar_datos()
    return pd.DataFrame(filas)

In [ ]:
borrar_log()
evals = await evaluar(casos, repeticiones=3)
evals

Resumen por caso:

In [ ]:
evals.assign(aciertos=evals.ok == "✅").groupby("correo").agg(
    aciertos=("aciertos", "mean"), coste_medio=("coste", "mean"),
).style.format({"aciertos": "{:.0%}", "coste_medio": "${:.4f}"})

Puntos clave:

- Se evalúa la **conducta** (qué herramientas usó), no solo la respuesta. Un agente puede responder `accion: "existente"` y haber creado igualmente un duplicado.
- Los agentes no son deterministas: cada caso se ejecuta varias veces y se informa del **porcentaje** de aciertos.
- Con cada cambio de prompt o de descripción, se ejecutan los evals: es la red de seguridad del método del 5.4. Por ejemplo, ¿sigue pasando todo con `SYSTEM_SEGURO`?
- Cada bug encontrado en producción se convierte en un caso nuevo de `casos.json`.

In [ ]:
evals_seguro = await evaluar(casos, repeticiones=1, options=crear_opciones(system_prompt=SYSTEM_SEGURO))
evals_seguro

## Cierre

El notebook es para aprender, explorar y depurar. Lo que va a producción es un módulo (`triaje.py`) con su `evaluar()` ejecutándose en CI. Ya lo hemos empezado a hacer en este vídeo.

### Checklist de producción

- [ ] Límites: `max_turns` y `max_budget_usd`, y el caso de error tratado
- [ ] Caché verificada (`cache_read_input_tokens > 0`)
- [ ] Solo las herramientas necesarias, sin herramientas genéricas
- [ ] Hooks de auditoría y de política
- [ ] La entrada externa tratada como dato
- [ ] Evals que se ejecutan en cada cambio
- [ ] Código del agente extraído del notebook a un módulo